# Inland Excess Water (IEW) Monitoring — Radar_Model (Stage B, second branch)

Implements the **binary S1-only** half of the paper's dual-model strategy (§3.2): *"Under cloudy conditions, we employ a radar-based model Radar_Model trained on S1 channels to detect waterlogged areas... a binary classification approach was adopted, where all water-affected classes were consolidated into a single 'water' class."* This model exists specifically to support **Temporal Gap-Filling** (`07_temporal_gapfilling.ipynb`), which needs a cloud-penetrating fallback prediction for days when the optical S2_Model can't see the ground.

**Input:** 3-channel Sentinel-1 (VV, VH, VV/VH ratio). **Output:** binary water-affected (raw classes 2-6 collapsed to one label) vs. not-affected (raw class 7); cloud/shadow pixels (10, 11) excluded from the loss exactly as in the S2 models (per §3.2: "these regions lack reliable supervision, they were excluded from the loss computation").

Paper §3.2 also states: *"Both models share an identical architecture, differing only in the number of input channels"* — so this notebook reuses the exact same RUPNet backbone as the S2 models, just with `in_channels=3` and `num_classes=2`.

**Pipeline position:** `A. Dataset → B. Dual-model training: S2_Model (unet/unetpp/rupnet.ipynb) + Radar_Model (this notebook) → ... → F. Temporal Gap-Filling`. This notebook has no dependency on Stage C/D/E and can run any time after Stage A.

Flow: imports → data pipeline → binary-label conversion → losses → model → dataloaders → training → loss curves → evaluation (binary Pr/Rc/F1/**Accuracy** - this is the one model in the pipeline where "accuracy" is actually a meaningful, well-defined metric, since it's a genuine 2-class problem) → prediction visualization.

## 1. Imports and Setup

In [ ]:
import os
import sys
import glob
import yaml
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
from rasterio.windows import Window
from pathlib import Path
from typing import List, Dict, Tuple

torch.manual_seed(0)
np.random.seed(0)

print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")


## 2. Data Pipeline
Same `FloodPatchDataset` as the S2 notebooks, but now with `load_s1=True` — the model consumes the S1 tensor and ignores S2.

In [ ]:
# --- Raster I/O helpers -----------------------------------------------------
def read_s2(path: str) -> np.ndarray:
    """Read a Sentinel-2 GeoTIFF -> (11, H, W) float32, scaled to reflectance [0, ~1.7]."""
    with rasterio.open(path) as src:
        data = src.read().astype(np.float32)
        data /= 10000.0  # S2 BOA reflectance is stored scaled by 10000
        return data

def read_s1(path: str) -> np.ndarray:
    """Read a Sentinel-1 GeoTIFF -> (3, H, W) float32: VV, VH, VV/VH ratio (dB)."""
    with rasterio.open(path) as src:
        return src.read().astype(np.float32)

def read_gt(path: str) -> np.ndarray:
    """Read a ground-truth GeoTIFF -> (H, W) int class map."""
    with rasterio.open(path) as src:
        return src.read(1)


def get_scene_files(dataset_root: str, split: str) -> List[Dict[str, str]]:
    """Find and align S1, S2, GT, GT_0_8 files for a split ('Train'/'Test'), keyed off S2."""
    root = Path(dataset_root)
    split_dir = root / "tifs" / split

    s1_dir = split_dir / "S1"
    s2_dir = split_dir / "S2"
    gt_dir = split_dir / "GT"
    gt08_dir = split_dir / "GT_0_8"

    s2_files = sorted(split_dir.joinpath("S2").glob("*.tif"))

    aligned = []
    for s2_file in s2_files:
        name = s2_file.name
        s1_path = s1_dir / name
        gt_path = gt_dir / name
        gt08_path = gt08_dir / name

        if not s1_path.exists():
            raise FileNotFoundError(f"Missing S1 file: {s1_path}")
        if not gt_path.exists():
            raise FileNotFoundError(f"Missing GT file: {gt_path}")
        if not gt08_path.exists():
            raise FileNotFoundError(f"Missing GT_0_8 file: {gt08_path}")

        aligned.append({
            "name": s2_file.stem,
            "S1": str(s1_path),
            "S2": str(s2_file),
            "GT": str(gt_path),
            "GT_0_8": str(gt08_path),
        })
    return aligned

print("Raster I/O helpers ready: read_s2, read_s1, read_gt, get_scene_files")


In [ ]:
# --- PyTorch Dataset ---------------------------------------------------------
# Per-channel statistics computed over the training split.
S2_MEAN = np.array([0.0436, 0.0557, 0.0723, 0.0838, 0.1151, 0.1598, 0.1800, 0.1974, 0.2004, 0.2233, 0.1706], dtype=np.float32)
S2_STD  = np.array([0.0182, 0.0233, 0.0272, 0.0340, 0.0380, 0.0598, 0.0686, 0.0751, 0.0728, 0.0674, 0.0606], dtype=np.float32)

S1_MEAN = np.array([-21.6081, -13.6686, -8.0403], dtype=np.float32)
S1_STD  = np.array([3.4706, 3.0644, 2.5309], dtype=np.float32)


class FloodPatchDataset(Dataset):
    """Sliding-window dataset over aligned S1/S2/GT scenes. Reads windows directly from
    disk via rasterio (or caches full scenes in memory), standardizes per-channel, and
    optionally applies flip/rotation augmentation.
    """

    def __init__(
        self,
        dataset_root: str,
        split: str,
        patch_size: int = 64,
        stride: int = 64,
        augment: bool = False,
        cache_in_memory: bool = False,
        load_s1: bool = True,
    ):
        self.dataset_root = dataset_root
        self.split = split
        self.patch_size = patch_size
        self.stride = stride
        self.augment = augment
        self.cache_in_memory = cache_in_memory
        self.load_s1 = load_s1

        self.scenes = get_scene_files(dataset_root, split)

        self.s2_cache, self.s1_cache, self.gt_cache = {}, {}, {}
        self.patches = []

        print(f"Initializing {split} dataset (cache_in_memory={cache_in_memory}, load_s1={load_s1})...")

        for scene_idx, scene in enumerate(self.scenes):
            with rasterio.open(scene["S2"]) as s2_src:
                H, W = s2_src.height, s2_src.width

            if cache_in_memory:
                with rasterio.open(scene["S2"]) as s2_src:
                    s2_img = s2_src.read().astype(np.float32)
                    s2_img /= 10000.0
                    self.s2_cache[scene_idx] = s2_img
                if load_s1:
                    with rasterio.open(scene["S1"]) as s1_src:
                        self.s1_cache[scene_idx] = s1_src.read().astype(np.float32)
                with rasterio.open(scene["GT_0_8"]) as gt_src:
                    self.gt_cache[scene_idx] = gt_src.read(1).astype(np.int64)
            # else: no persistent file handles are opened here - see __getitem__, which
            # opens+reads+closes per access. This keeps the Dataset free of unpicklable
            # state (required for num_workers > 0, since DataLoader workers get a pickled
            # copy of the Dataset) and avoids GDAL's internal per-handle cache growing
            # unbounded over a long-running, many-epoch loop that reuses the same handle.

            for y in range(0, H - patch_size + 1, stride):
                for x in range(0, W - patch_size + 1, stride):
                    self.patches.append({"scene_idx": scene_idx, "y": y, "x": x})

        print(f"Loaded {split} dataset with {len(self.patches)} total patches from {len(self.scenes)} scenes.")

    def __len__(self) -> int:
        return len(self.patches)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        info = self.patches[idx]
        scene_idx, y, x = info["scene_idx"], info["y"], info["x"]

        if self.cache_in_memory:
            s2_data = self.s2_cache[scene_idx][:, y:y + self.patch_size, x:x + self.patch_size].copy()
            if self.load_s1:
                s1_data = self.s1_cache[scene_idx][:, y:y + self.patch_size, x:x + self.patch_size].copy()
            else:
                s1_data = np.zeros((3, self.patch_size, self.patch_size), dtype=np.float32)
            gt_data = self.gt_cache[scene_idx][y:y + self.patch_size, x:x + self.patch_size].copy()
        else:
            scene = self.scenes[scene_idx]
            window = Window(x, y, self.patch_size, self.patch_size)
            with rasterio.open(scene["S2"]) as s2_src:
                s2_data = s2_src.read(window=window).astype(np.float32)
            s2_data /= 10000.0
            if self.load_s1:
                with rasterio.open(scene["S1"]) as s1_src:
                    s1_data = s1_src.read(window=window).astype(np.float32)
            else:
                s1_data = np.zeros((3, self.patch_size, self.patch_size), dtype=np.float32)
            with rasterio.open(scene["GT_0_8"]) as gt_src:
                gt_data = gt_src.read(1, window=window).astype(np.int64)

        for c in range(11):
            s2_data[c] = (s2_data[c] - S2_MEAN[c]) / S2_STD[c]
        if self.load_s1:
            for c in range(3):
                s1_data[c] = (s1_data[c] - S1_MEAN[c]) / S1_STD[c]

        if self.augment:
            if np.random.rand() > 0.5:
                s2_data = np.flip(s2_data, axis=2).copy()
                if self.load_s1:
                    s1_data = np.flip(s1_data, axis=2).copy()
                gt_data = np.flip(gt_data, axis=1).copy()
            if np.random.rand() > 0.5:
                s2_data = np.flip(s2_data, axis=1).copy()
                if self.load_s1:
                    s1_data = np.flip(s1_data, axis=1).copy()
                gt_data = np.flip(gt_data, axis=0).copy()
            rot_k = np.random.randint(0, 4)
            if rot_k > 0:
                s2_data = np.rot90(s2_data, k=rot_k, axes=(1, 2)).copy()
                if self.load_s1:
                    s1_data = np.rot90(s1_data, k=rot_k, axes=(1, 2)).copy()
                gt_data = np.rot90(gt_data, k=rot_k, axes=(0, 1)).copy()

        return torch.from_numpy(s2_data), torch.from_numpy(s1_data), torch.from_numpy(gt_data)


print("FloodPatchDataset ready")


In [ ]:
def stretch_rgb(rgb: np.ndarray, low: float = 2.0, high: float = 98.0) -> np.ndarray:
    """Per-channel percentile contrast stretch for a true-color composite (see rupnet.ipynb
    diagnosis note - a fixed divisor washes scenes out or leaves them too dark)."""
    out = np.empty_like(rgb, dtype=np.float32)
    for c in range(rgb.shape[-1]):
        lo, hi = np.percentile(rgb[..., c], [low, high])
        out[..., c] = np.clip((rgb[..., c] - lo) / (hi - lo + 1e-8), 0, 1)
    return out


print("stretch_rgb ready")


## 3. Binary Label Conversion
`GT_0_8` encodes the 6 base IEW classes as 0-4 (water/soil/vegetation severity, i.e. raw classes 2-6) and 5 ("not affected", raw class 7); cloud/shadow is 6 (ignored). For the Radar_Model, classes 0-4 collapse to a single "water-affected" label (1) and class 5 stays "not affected" (0); the ignore index shifts from 6 to 2.

In [ ]:
def to_binary_target(gt: torch.Tensor) -> torch.Tensor:
    """0-4 (water/soil/veg severity) -> 1 ("affected"); 5 (not affected) -> 0; 6 (cloud/shadow,
    ignored) stays as ignore_index=2 for the binary loss functions below."""
    out = torch.full_like(gt, 2)  # default: ignore
    out[gt < 5] = 1
    out[gt == 5] = 0
    return out


print("to_binary_target ready")


## 4. Loss Functions
Same GDL + WFL, reused with `num_classes=2, ignore_index=2`.

In [ ]:
class GeneralizedDiceLoss(nn.Module):
    """Class-weighted Generalized Dice Loss (Sudre et al., 2017), with an ignore_index
    to mask out cloud/shadow pixels (label >= ignore_index) from the loss."""

    def __init__(self, ignore_index: int = 6, num_classes: int = 6, eps: float = 1e-6):
        super().__init__()
        self.ignore_index = ignore_index
        self.num_classes = num_classes
        self.eps = eps

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=1)

        valid_mask = (targets < self.ignore_index).unsqueeze(1)
        probs = probs * valid_mask

        targets_clamped = torch.clamp(targets, 0, self.num_classes - 1)
        targets_onehot = F.one_hot(targets_clamped, num_classes=self.num_classes)
        targets_onehot = targets_onehot.permute(0, 3, 1, 2).float()
        targets_onehot = targets_onehot * valid_mask

        intersection = torch.sum(probs * targets_onehot, dim=(0, 2, 3))
        union = torch.sum(probs + targets_onehot, dim=(0, 2, 3))

        class_pixel_counts = torch.sum(targets_onehot, dim=(0, 2, 3))
        weights = 1.0 / (class_pixel_counts ** 2 + self.eps)
        weights = weights / (torch.sum(weights) + self.eps)

        numerator = 2.0 * torch.sum(weights * intersection)
        denominator = torch.sum(weights * union)

        return 1.0 - (numerator + self.eps) / (denominator + self.eps)


class WeightedFocalLoss(nn.Module):
    """Class-weighted Focal Loss (Lin et al., 2017) with an ignore_index."""

    def __init__(self, alpha: torch.Tensor = None, gamma: float = 2.0, ignore_index: int = 6):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ignore_index = ignore_index

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        C = logits.size(1)
        logits_flat = logits.permute(0, 2, 3, 1).contiguous().view(-1, C)
        targets_flat = targets.view(-1)

        valid_mask = targets_flat < self.ignore_index
        if not valid_mask.any():
            return torch.tensor(0.0, device=logits.device, requires_grad=True)

        logits_valid = logits_flat[valid_mask]
        targets_valid = targets_flat[valid_mask]

        log_probs = F.log_softmax(logits_valid, dim=1)
        log_probs_target = log_probs[torch.arange(len(targets_valid)), targets_valid]
        probs_target = torch.exp(log_probs_target)

        focal_weight = (1.0 - probs_target) ** self.gamma
        loss = -focal_weight * log_probs_target

        if self.alpha is not None:
            self.alpha = self.alpha.to(logits.device)
            loss = loss * self.alpha[targets_valid]

        return loss.mean()


print("GeneralizedDiceLoss, WeightedFocalLoss ready")


## 5. Model Architecture — RUPNet (3-channel input, 2-class output)
Same backbone family used for the S2 models, so the whole pipeline shares one architecture end to end (paper §3.2's stated rationale: S2_Model and Radar_Model differ only in input channel count).

In [ ]:
class Conv2D(nn.Module):
    def __init__(self, in_c, out_c, kernel_size=3, padding=1, stride=1, dilation=1, bias=False, act=True):
        super().__init__()
        self.act = act
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size, stride=stride, padding=padding, dilation=dilation, bias=bias),
            nn.BatchNorm2d(out_c),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        if self.act:
            x = self.relu(x)
        return x


class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
        )
        self.shortcut = nn.Sequential()
        if in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=1, stride=1, bias=False),
                nn.BatchNorm2d(out_c),
            )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.conv(x) + self.shortcut(x))


class DecoderBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.res = ResidualBlock(in_c + out_c, out_c)

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        return self.res(x)


class UpBlock(nn.Module):
    def __init__(self, in_c, out_c, scale):
        super().__init__()
        self.up = nn.Upsample(scale_factor=scale, mode="bilinear", align_corners=True)
        self.res = ResidualBlock(in_c, out_c)

    def forward(self, x):
        return self.res(self.up(x))


class RUPNet(nn.Module):
    """RUPNet (Tomar, Bagci & Jha, 2023) - the paper's winning backbone (Table 1,
    F1 = 81.19%) and its knowledge-distillation teacher/student architecture (~2.05M params
    at in_channels=11; scales down automatically for fewer input channels, e.g. the 3-channel
    Radar_Model)."""

    def __init__(self, in_channels: int = 11, num_classes: int = 6):
        super().__init__()

        self.init_conv = Conv2D(in_channels, 32, kernel_size=3, padding=1)

        self.enc1_1 = ResidualBlock(32, 64)
        self.down1 = nn.MaxPool2d(2)
        self.enc1_2 = ResidualBlock(64, 64)
        self.down2 = nn.MaxPool2d(2)  # scale: H/4

        self.enc2 = ResidualBlock(64, 128)
        self.down3 = nn.MaxPool2d(2)  # scale: H/8

        self.enc3 = ResidualBlock(128, 256)
        self.down4 = nn.MaxPool2d(2)  # scale: H/16

        self.c1 = Conv2D(64, 64, kernel_size=1, padding=0)
        self.c2 = Conv2D(128, 64, kernel_size=1, padding=0)
        self.c3 = Conv2D(256, 64, kernel_size=1, padding=0)

        self.d1 = DecoderBlock(64, 64)
        self.d2 = DecoderBlock(64, 64)

        self.d3 = UpBlock(64, 64, scale=4)
        self.u1 = UpBlock(64, 64, scale=4)
        self.u2 = UpBlock(64, 64, scale=8)
        self.u3 = UpBlock(64, 64, scale=16)

        self.fusion = ResidualBlock(256, 64)
        self.outc = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, inputs):
        x = self.init_conv(inputs)

        e1 = self.enc1_1(x)
        e1 = self.down1(e1)
        e1 = self.enc1_2(e1)
        e1 = self.down2(e1)  # (B, 64, H/4, W/4)

        e2 = self.enc2(e1)
        e2 = self.down3(e2)  # (B, 128, H/8, W/8)

        e3 = self.enc3(e2)
        e3 = self.down4(e3)  # (B, 256, H/16, W/16)

        c1 = self.c1(e1)
        c2 = self.c2(e2)
        c3 = self.c3(e3)

        d1 = self.d1(c3, c2)  # (B, 64, H/8, W/8)
        d2 = self.d2(d1, c1)  # (B, 64, H/4, W/4)

        d3 = self.d3(d2)  # (B, 64, H, W)
        u1 = self.u1(c1)  # (B, 64, H, W)
        u2 = self.u2(c2)  # (B, 64, H, W)
        u3 = self.u3(c3)  # (B, 64, H, W)

        fused = torch.cat([d3, u1, u2, u3], dim=1)  # (B, 256, H, W)
        fused = self.fusion(fused)  # (B, 64, H, W)

        return self.outc(fused)


print("RUPNet ready")


## 6. Dataset & Dataloader Setup
Full non-overlapping patch grid (100% spatial coverage, ~52k patches) built once; each epoch trains on a fresh random subset of it (§8) rather than the whole grid or a shrunk static grid - see `rupnet.ipynb` §6 for the full reasoning, including two rejected alternatives.

In [ ]:
with open("configs/data.yaml", "r") as f:
    config = yaml.safe_load(f)
dataset_root = config["dataset_root"]

# Binary class weights from training-set pixel frequency: water-affected (raw 2-6) vs.
# not-affected (raw 7). Reuses the same per-class pixel counts as the S2 notebooks.
TRAIN_PIXEL_COUNTS_6 = [7674525, 651441, 900550, 4493763, 3450129, 195012790]
water_pixels = sum(TRAIN_PIXEL_COUNTS_6[:5])
notaffected_pixels = TRAIN_PIXEL_COUNTS_6[5]
counts = np.array([notaffected_pixels, water_pixels], dtype=np.float32)  # [class0, class1]
weights = counts.sum() / (2 * counts)
weights = weights / weights.mean()
class_weights = torch.from_numpy(weights).float()
print(f"Binary class weights [not-affected, water-affected]: {class_weights.tolist()}")

BATCH_SIZE = 32  # paper §4.1

train_dataset = FloodPatchDataset(
    dataset_root=dataset_root, split="Train", patch_size=config["patch_size"],
    stride=config["patch_size"], augment=True, cache_in_memory=False, load_s1=True,
)
test_dataset = FloodPatchDataset(
    dataset_root=dataset_root, split="Test", patch_size=config["patch_size"],
    stride=512, augment=False, cache_in_memory=False, load_s1=True,
)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Full train patch grid: {len(train_dataset):,} patches (100% spatial coverage)")


## 7. Model, Loss, and Optimizer Setup
Optimizer/LR/epochs match the paper's §4.1 Implementation Details verbatim (`Adam`, LR `1e-4`, `100` epochs, batch `32`) — the paper does not give separate hyperparameters for the Radar_Model, so the same stated recipe used for the S2 backbones is extended here, clearly labeled as such rather than silently assumed.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = RUPNet(in_channels=3, num_classes=2).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Radar_Model (RUPNet, 3ch->2class) params={n_params:,}")

EPOCHS = 100
LEARNING_RATE = 1e-4

criterion_gdl = GeneralizedDiceLoss(ignore_index=2, num_classes=2)
criterion_wfl = WeightedFocalLoss(alpha=class_weights, gamma=2.0, ignore_index=2)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


## 8. Training Loop
Train for 100 epochs at a constant LR (paper §4.1). Each epoch trains on a fresh random subset (`SAMPLES_PER_EPOCH` patches) of the full ~52k-patch grid - see §6. Tracks GDL/WFL/total loss and saves the best checkpoint by validation loss.

In [ ]:
epochs = EPOCHS
SAMPLES_PER_EPOCH = 13000  # ~1/4 of the full grid per epoch, redrawn randomly every epoch
                            # so there is no permanent spatial blind spot (see §6)
epoch_rng = np.random.default_rng(0)

history = {"train_loss": [], "val_loss": [], "train_gdl": [], "train_wfl": []}
best_val_loss = float("inf")

for epoch in range(1, epochs + 1):
    model.train()
    epoch_loss = epoch_gdl = epoch_wfl = 0.0

    # Fresh random subset of the full patch grid this epoch - see §6/§8 notes.
    subset_idx = epoch_rng.choice(len(train_dataset), size=min(SAMPLES_PER_EPOCH, len(train_dataset)), replace=False)
    train_loader = DataLoader(Subset(train_dataset, subset_idx), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

    for _, s1_patches, gt_patches in train_loader:
        s1_patches = s1_patches.to(device)
        gt_bin = to_binary_target(gt_patches).to(device)

        optimizer.zero_grad()
        logits = model(s1_patches)
        loss_gdl = criterion_gdl(logits, gt_bin)
        loss_wfl = criterion_wfl(logits, gt_bin)
        loss = loss_gdl + loss_wfl
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_gdl += loss_gdl.item()
        epoch_wfl += loss_wfl.item()

    n_batches = len(train_loader)
    avg_train_loss, avg_train_gdl, avg_train_wfl = epoch_loss / n_batches, epoch_gdl / n_batches, epoch_wfl / n_batches

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for _, s1_patches, gt_patches in test_loader:
            s1_patches = s1_patches.to(device)
            gt_bin = to_binary_target(gt_patches).to(device)
            logits = model(s1_patches)
            loss_gdl = criterion_gdl(logits, gt_bin)
            loss_wfl = criterion_wfl(logits, gt_bin)
            val_loss += (loss_gdl + loss_wfl).item()

    avg_val_loss = val_loss / len(test_loader)

    history["train_loss"].append(avg_train_loss)
    history["train_gdl"].append(avg_train_gdl)
    history["train_wfl"].append(avg_train_wfl)
    history["val_loss"].append(avg_val_loss)

    is_best = avg_val_loss < best_val_loss
    if is_best:
        best_val_loss = avg_val_loss
        torch.save({"model_state_dict": model.state_dict(), "epoch": epoch, "val_loss": avg_val_loss},
                   "outputs/checkpoints/radar_model_best.pt")

    flag = " *" if is_best else ""
    print(f"Epoch {epoch}/{epochs} - Train Loss: {avg_train_loss:.4f} (GDL: {avg_train_gdl:.4f}, WFL: {avg_train_wfl:.4f}) "
          f"| Val Loss: {avg_val_loss:.4f}{flag}")

print(f"Training complete! Best val loss: {best_val_loss:.4f} -> outputs/checkpoints/radar_model_best.pt")


## 9. Loss Curves

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history["train_loss"], label="Train Total Loss")
plt.plot(history["train_gdl"], label="Train GDL")
plt.plot(history["train_wfl"], label="Train WFL")
plt.plot(history["val_loss"], label="Val Total Loss", linestyle="--")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Radar_Model Loss Curves (GDL + WFL)")
plt.legend(); plt.grid(True); plt.show()


## 10. Model Evaluation on Full Test Set
Because this is a genuine 2-class problem, **pixel accuracy is a well-defined, meaningful number here** — unlike the 6-class S2 models, where a raw "overall accuracy" is dominated by the huge, easy "not affected" majority class (see the diagnosis note in `rupnet.ipynb` §0 and the "paper-comparable metric" cells in the S2 notebooks).

In [ ]:
eval_dataset = FloodPatchDataset(
    dataset_root=dataset_root, split="Test", patch_size=config["patch_size"],
    stride=config["patch_size"], augment=False, cache_in_memory=False, load_s1=True,
)
eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model.eval()
tp = fp = fn = tn = 0
correct_pixels = 0
total_pixels = 0

with torch.no_grad():
    for _, s1_patches, gt_patches in eval_loader:
        s1_patches = s1_patches.to(device)
        gt_bin = to_binary_target(gt_patches).numpy()
        logits = model(s1_patches)
        preds = torch.argmax(logits, dim=1).cpu().numpy()

        valid_mask = gt_bin < 2
        pred_water = (preds == 1) & valid_mask
        gt_water = (gt_bin == 1) & valid_mask

        tp += np.sum(pred_water & gt_water)
        fp += np.sum(pred_water & ~gt_water)
        fn += np.sum(~pred_water & gt_water)
        tn += np.sum(~pred_water & ~gt_water & valid_mask)
        correct_pixels += np.sum((preds == gt_bin) & valid_mask)
        total_pixels += np.sum(valid_mask)

pr = tp / (tp + fp + 1e-10)
rc = tp / (tp + fn + 1e-10)
f1 = 2 * pr * rc / (pr + rc + 1e-10)
accuracy = correct_pixels / total_pixels

print("=" * 60)
print(f"Radar_Model (binary water / non-water) - full test set")
print("=" * 60)
print(f"Precision (water class): {pr*100:.2f}%")
print(f"Recall    (water class): {rc*100:.2f}%")
print(f"F1        (water class): {f1*100:.2f}%")
print(f"Pixel Accuracy (both classes): {accuracy*100:.2f}%")
print("=" * 60)
print(f"Confusion: TP={tp:,}  FP={fp:,}  FN={fn:,}  TN={tn:,}")


## 11. Visualizing Predictions

In [ ]:
def patch_water_fraction(idx):
    _, s1, gt = eval_dataset[idx]
    gt_bin = to_binary_target(gt).numpy()
    valid = gt_bin[gt_bin < 2]
    return valid.mean() if valid.size else 0.0

# Pick patches with a healthy mix of both classes (fraction closest to 0.5), rather than
# arbitrary fixed indices, for an informative qualitative comparison.
scan_indices = range(0, len(eval_dataset), 37)
scored = sorted(((idx, abs(patch_water_fraction(idx) - 0.5)) for idx in scan_indices), key=lambda t: t[1])
chosen = [idx for idx, _ in scored[:3]]

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for i, patch_idx in enumerate(chosen):
    _, s1, gt = eval_dataset[patch_idx]
    s1_in = s1.unsqueeze(0).to(device)
    with torch.no_grad():
        pred = torch.argmax(model(s1_in), dim=1).squeeze(0).cpu().numpy()
    gt_bin = to_binary_target(gt).numpy()

    vv = s1.numpy()[0] * S1_STD[0] + S1_MEAN[0]

    axes[i, 0].imshow(vv, cmap="gray"); axes[i, 0].set_title("Sentinel-1 VV (dB)"); axes[i, 0].axis("off")
    axes[i, 1].imshow(gt_bin, cmap="Blues", vmin=0, vmax=1); axes[i, 1].set_title("GT (binary)"); axes[i, 1].axis("off")
    axes[i, 2].imshow(pred, cmap="Blues", vmin=0, vmax=1); axes[i, 2].set_title("Radar_Model Prediction"); axes[i, 2].axis("off")

plt.tight_layout()
plt.show()


## 12. Summary — fill in after running

- **Test accuracy:** ___  **F1 (water class):** ___
- **Checkpoint produced:** `outputs/checkpoints/radar_model_best.pt` — this is the model `07_temporal_gapfilling.ipynb` expects for its cloud-penetrating fallback prediction.
- Note: this binary model is expected to have *lower resolution* information than the S2 severity models by design (§3.2) — it cannot distinguish soil moisture severity from vegetation water stress, only "water-affected vs. not." That's fine; its only job in the pipeline is filling in *where* something is wrong under clouds, not *how badly*.